## Load libraries

In [ ]:
import polars as pl
import os
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
import polars as pl
pl.Config.set_tbl_rows(20)  # show up to 20 rows
import polars.selectors as cs
import seaborn as sns
sns.set_style("darkgrid")  # darkgrid
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

In [ ]:
def csv_to_parquet(path):
    os.makedirs("raw_data", exist_ok=True)
    filename = os.path.basename(path)
    parquet_filename = filename.replace(".csv", ".parquet")
    parquet_path = os.path.join(".\\raw_data", parquet_filename)
    # lazy scan (does NOT load everything into RAM)
    lf = pl.scan_csv(
        path,
        infer_schema_length=1000,
        ignore_errors=True
    )
    print(parquet_path)
    # stream directly to parquet
    lf.sink_parquet(
        parquet_path,
        compression="snappy"
    )
    print(f"Saved Parquet file to: {parquet_path}")


def parse_lc_date(col_name: str) -> pl.Expr:
    """Parse LendingClub date strings of the form 'Dec-2015' into pl.Date.

    Safe if the column is already a date type — the Utf8 cast is non-strict
    and returns null on incompatible types, which downstream code handles.
    """
    return (
        pl.col(col_name)
        .cast(pl.Utf8, strict=False)
        .str.to_date(format="%b-%Y", strict=False)
    )

## Convert pandas to polar

In [ ]:
has_parquet = any(f.endswith(".parquet") for f in os.listdir("../data/raw_data"))

if has_parquet:
    print("Parquet file already exists in data folder")
else:
    print("No parquet file found. Converting CSV → Parquet...")
    csv_to_parquet("../data/raw_data/accepted_2007_to_2018Q4.csv")

## Read accepted_df

In [ ]:
# --- Use polar for faster processing ---
accepted_df = pl.read_parquet("../data/raw_data/accepted_2007_to_2018Q4.parquet")
# Parse date strings into actual Date type for arithmetic
accepted_df = accepted_df.with_columns([
        parse_lc_date("issue_d").alias("issue_d_parsed"),
        parse_lc_date("earliest_cr_line").alias("earliest_cr_line_parsed"),
    ])
accepted_df.head()

In [ ]:
# --- Dataset schema ---
schema = accepted_df.schema
# null counts (computed once)
null_counts = accepted_df.null_count().row(0)
# total rows
n_rows = accepted_df.height

# build summary dataframe efficiently
type_summary = pl.DataFrame({
    "column": accepted_df.columns,
    "dtype": [str(schema[c]) for c in accepted_df.columns],
    "nulls": list(null_counts),
})

# compute percentages
type_summary = (
    type_summary
    .with_columns(
        (
            pl.col("nulls") / n_rows * 100
        ).round(2).alias("null_pct")
    )
    .sort("dtype")
)

print(f"Dataset shape: {accepted_df.shape}")
print("\n----- Null Values Distribution -----")
# avoid printing gigantic dataframe
print(type_summary)

## Data understanding

In [ ]:
# --- Drop missing dates ---
n_null_issue = (
    accepted_df
    .select(pl.col("issue_d_parsed").is_null().sum())
    .item()
)
print(f"Rows with null issue_d_parsed: {n_null_issue}")

# drop nulls directly
accepted_df = accepted_df.drop_nulls(subset=["issue_d_parsed"])

print("Remove null in column issue_d_parsed done!")
print(f"New shape: {accepted_df.shape}")

In [ ]:
# Polar String display
pl.Config.set_fmt_str_lengths(1000)

# Loan Status Distribution
print(f'Total Rows    : {accepted_df.shape[0]:,}')
print(f'Total Columns : {accepted_df.shape[1]}')
date_range = accepted_df.select(
    pl.col("issue_d_parsed")
      .min().alias("min_date"),
    pl.col("issue_d_parsed")
      .max().alias("max_date"),
)

print(
    f"Date Range: "
    f"{date_range[0, 'min_date']:%b-%Y} → "
    f"{date_range[0, 'max_date']:%b-%Y}"
)

print('\n----- Loan Status Distribution -----')
print(accepted_df['loan_status'].value_counts().sort("count", descending=True))

## General info

In [ ]:
# Keep only Fully Paid and Charged Off
accepted_df = accepted_df.filter(
    pl.col("loan_status").is_in(["Fully Paid", "Charged Off"])
)

# Create binary target
accepted_df = accepted_df.with_columns(
    pl.when(pl.col("loan_status") == "Charged Off")
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias("target")
)

# Compute default rate
default_rate = (
    accepted_df
    .select(pl.col("target").mean())
    .item()
    * 100
)

print(f"Dataset columns: {accepted_df.columns}")
print(f"Overall Default Rate: {default_rate:.2f}%")

## Data Cleaning

In [ ]:
# Merge rare home_ownership
clean_df = accepted_df.with_columns(
    pl.col('home_ownership').replace({'NONE': 'OTHER', 'ANY': 'OTHER'})
)

# extract integer in sub_grade
clean_df = clean_df.with_columns(
    pl.col("sub_grade").str.extract(r'(\d+)').cast(pl.Int8)

)
# term: "36 months"  -> int months (needed for interest burden)
clean_df = clean_df.with_columns(
        pl.col("term").cast(pl.Utf8).str.extract(r"(\d+)").cast(pl.Int16)
)

# emp_length → integer
clean_df = clean_df.with_columns(
    pl.col('emp_length')
    .str.replace('10\\+ years', '10 years')
    .str.replace('< 1 year', '0 years')
    .str.extract(r'(\d+)')
    .cast(pl.Int8)
    .alias('emp_length')
)


# --- Impute missing values ---
clean_df = clean_df.with_columns([
        pl.col("emp_length").fill_null(pl.col("emp_length").median()),
        pl.col("percent_bc_gt_75").fill_null(pl.col("percent_bc_gt_75").median()),
        pl.col("mort_acc").fill_null(pl.col("mort_acc").median()),
        pl.col("dti").fill_null(pl.col("dti").median()),
        pl.col("bc_util").fill_null(pl.col("bc_util").median()),
        pl.col("revol_util").fill_null(pl.col("revol_util").median()),
        pl.col("tot_cur_bal").fill_null(pl.col("tot_cur_bal").median()),
        pl.col("pub_rec").fill_null(0),
        pl.col("inq_last_6mths").fill_null(0),
        pl.col("tax_liens").fill_null(0),
        pl.col("acc_now_delinq").fill_null(0)
])

## Feature Selection

1. Select features has null counts less than 10% of feature observations.
2. Feature correlation
3. Mutual information of X and y

In [ ]:
# --- Polars missing value analysis ---
missing_df = (
    pl.DataFrame({
        "column": accepted_df.columns,
        "null_count": [accepted_df[col].is_null().sum() for col in accepted_df.columns]
    })
    .with_columns(
        (pl.col("null_count") / len(accepted_df) * 100).alias("null_pct")
    )
    .filter(pl.col("null_pct") > 0)
    .sort("null_pct", descending=True)
)

# --- Select variables has < 10% null values
missing_lt_10 = missing_df.filter(
                (pl.col("null_pct") < 10) 
             )

print(f"Selected columns which is < 10% missing values:")
print(accepted_df.select([col for col in accepted_df.columns
                          if (col in missing_lt_10["column"])]).columns)

In [ ]:
FEATURES_1 = [
    # Loan info
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 
    'sub_grade', 'purpose', 'funded_amnt', 'funded_amnt_inv',
     # Borrower financials
    'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    # Employment
    'emp_length', 'home_ownership',
    # Credit history
    'earliest_cr_line_parsed', 'fico_range_low', 'fico_range_high',
    'tot_cur_bal', 'total_rev_hi_lim',
    # Risk
    'verification_status', 'tax_liens',
    'acc_now_delinq', 'mort_acc', 'bc_util', 'percent_bc_gt_75',
    # Target
    'issue_d_parsed', 'target'
]

# Filter to only existing columns (safety check)
available = [col for col in FEATURES_1 if col in accepted_df.columns]
df = clean_df.select(available)
print(f"Selected {len(available) - 1} features + target")
print(f"Shape: {df.shape}")
print(df.null_count())

### Build heatmap - correlated features

In [ ]:
def feat_corr(df: pl.DataFrame, features: list) -> None:
    """
    Computes and plots a correlation heatmap for all numeric features.

    Args:
        df       : Polars DataFrame
        features : list of feature column names
    """

    # Select only numeric columns
    numeric_df = df.select(features).select(cs.numeric()).to_pandas()
    non_numeric = [c for c in features if c not in numeric_df.columns]

    print(f"Numeric features   : {len(numeric_df.columns)}")
    print(f"Non-numeric features)  : {non_numeric}")

    # Compute correlation matrix
    corr_matrix = numeric_df.corr()

    # Plot
    n = len(corr_matrix)
    fig, ax = plt.subplots(figsize=(n * 0.7 + 2, n * 0.6 + 2))

    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # upper triangle mask

    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 7},
        cmap="RdBu_r",
        center=0,
        vmin=-1, vmax=1,
        linewidths=0.4,
        linecolor="white",
        square=True,
        cbar_kws={"shrink": 0.6, "label": "Pearson r"},
        ax=ax
    )

    ax.set_title("Feature Correlation Matrix (numeric only)",
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

    plt.tight_layout()
    plt.savefig("../img/correlation_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Top 10 correlated pairs (excluding self-correlation)
    corr_pairs = (
        corr_matrix
        .unstack()
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "correlation"})
        .query("feature_1 != feature_2")                        # drop self
        .assign(abs_corr=lambda x: x["correlation"].abs())
        .drop_duplicates(subset=["abs_corr"])                   # drop mirror pairs
        .sort_values("abs_corr", ascending=False)
        .head(10)
    )

    print("\nTop 10 most correlated pairs:")
    print(corr_pairs[["feature_1", "feature_2", "correlation"]].to_string(index=False))

# Feature correlation
feat_corr(df, FEATURES_1)

Highly correlated features need to be removed: `fico_range_high`, `funded_amnt`, `funded_amnt_inv`, `installment`.

### Mutual Information

In [ ]:
# Remove `fico_range_high`, `funded_amnt`, `funded_amnt_inv`, `installment`
FEATURES_2 = [
    # Loan info
    'loan_amnt', 'term', 'int_rate', 'grade', 
    'sub_grade', 'purpose', 
     # Borrower financials
    'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'total_acc', 'pub_rec', 'revol_bal', 'revol_util',
    # Employment
    'emp_length', 'home_ownership',
    # Credit history
    'earliest_cr_line_parsed', 'fico_range_low', 'tot_cur_bal', 'total_rev_hi_lim',
    # Risk
    'verification_status', 'tax_liens', 'acc_now_delinq', 
    'mort_acc', 'bc_util', 'percent_bc_gt_75',
    # Target
    'issue_d_parsed', 'target'
]

In [ ]:
# --- Separate features and target ---
info_df = df.select(FEATURES_2)
info_df = info_df.drop_nulls()
X = info_df.drop('target')
y = info_df['target'] 

# 2. Identify categorical/string columns using Polars Selectors
cat_cols = X.select(cs.string() | cs.categorical()).columns

# Ordinal Encoding string
X = X.with_columns(cs.string().cast(pl.Categorical).to_physical())

# Create the boolean mask matching the exact column order of X
discrete_features_mask = [col in cat_cols for col in X.columns]

# 3. Calculate Mutual Information
print("Calculating Mutual Information (this may take a moment)...")
# We convert X and y to numpy arrays since scikit-learn doesn't natively speak Polars
mi_scores = mutual_info_classif(
    X.to_numpy(), 
    y.to_numpy(), 
    discrete_features=discrete_features_mask, 
    random_state=42
)

# 4. Format results back into a Polars DataFrame
mi_results = pl.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
}).sort('MI_Score', descending=True)

# 5. Visualize
# Modern Seaborn accepts Polars DataFrames natively!
plt.figure(figsize=(10, 8))
sns.barplot(x='MI_Score', y='Feature', data=mi_results, palette='Oranges_d')
plt.title("Mutual Information Scores (Higher means more predictive power)")
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

## Final features + derived
* remove correlated features appear in top 10 pairs: 'fico_range_high', 'funded_amnt', 'funded_amnt_inv' 
* remove leakage features: 'last_pymnt_amnt',  "total_pymnt"

In [ ]:
FEATURES_FINAL = [
    # 1. Loan Characteristics (The Application)
    'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'purpose',
    # 2. Borrower Demographics & Financials (The Person)
    'annual_inc', 'emp_length', 'home_ownership', 'dti',
    # 3. Credit History & Utilization (The Track Record)
    'fico_range_low', 'earliest_cr_line_parsed', 'open_acc', 'total_acc',
    'mort_acc', 'tot_cur_bal', 'revol_bal', 'revol_util',
    'bc_util', 'percent_bc_gt_75', 'inq_last_6mths',
    # 4. Red Flags
    'delinq_2yrs', 'acc_now_delinq', 'tax_liens', 'pub_rec',
    # 5. Metadata & Target Variables
    'issue_d_parsed', 'target'
]

In [ ]:
final_df = df.select(FEATURES_FINAL)

# ---- Derived columns ----
# Income / debt-burden ratios
final_df = final_df.with_columns([
    (pl.col("loan_amnt") / (pl.col("annual_inc") + 1))
        .alias("loan_to_income"),
    (pl.col("revol_bal") / (pl.col("annual_inc") + 1))
        .alias("revolbal_to_income"),
    (pl.col("loan_amnt") * pl.col("int_rate") / (12 * pl.col("annual_inc") + 1))
        .alias("payment_to_income"),
]) 

# Credit behavior 
final_df = final_df.with_columns([
    (pl.col("inq_last_6mths") / (pl.col("total_acc") + 1))
        .alias("inquiries_per_account"),
    (pl.col("revol_util") * pl.col("dti")).alias("revol_stress"),
    (pl.col("open_acc") / (pl.col("total_acc") + 1)).alias("active_accounts_ratio")
])

# Loan-structure
cfinal_df = final_df.with_columns([
    (pl.col("int_rate") * (pl.col("term") / 12.0))
        .alias("total_interest_burden"),
])

# Credit history length 
final_df = final_df.with_columns([
    ((pl.col("issue_d_parsed") - pl.col("earliest_cr_line_parsed"))
        .dt.total_days() / 365.25).alias("credit_history_years")
    ])
final_df = final_df.with_columns([
    pl.col("credit_history_years").fill_null(pl.col("credit_history_years").median()),
])

# Red flags occurrences
final_df = final_df.with_columns([
    (
        (pl.col("pub_rec") > 0).cast(pl.Int32) +
        (pl.col("tax_liens") > 0).cast(pl.Int32) +
        (pl.col('delinq_2yrs') > 0).cast(pl.Int32) +
        (pl.col('acc_now_delinq') > 0).cast(pl.Int32))
        .alias("total_red_flags")
])

In [ ]:
null_summary = (
    final_df
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .filter(pl.col("null_count") > 0)
)

print(null_summary)

In [ ]:
print(f"Data shape: {final_df.shape}")     
print(f"Number of null columns: {final_df.null_count()}")